# College Football Upset Vulnerability — Stage 3: Eligibility and Temporal Splits


Historical subdivision source: [SportsDataverse team-group membership](https://cfbfastr.sportsdataverse.org/reference/load_cfb_team_group_seasons.html)

## 0. Mount Drive and load the Stage 2 tables

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import re
import unicodedata
from difflib import get_close_matches

import numpy as np
import pandas as pd

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/CFB_Upset_Project')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
REFERENCE_DIR = PROJECT_DIR / 'data' / 'reference'
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

team_games = pd.read_csv(PROCESSED_DIR / 'team_games.csv', parse_dates=['date'])
matchups = pd.read_csv(PROCESSED_DIR / 'matchup_features_initial.csv', parse_dates=['date'])

assert team_games['game_id'].nunique() == matchups['game_id'].nunique() == 18_300
assert len(team_games) == 2 * len(matchups)
assert matchups['game_id'].is_unique
assert set(range(2004, 2026)) == set(matchups['season'].unique())

print(f'Team-game rows: {len(team_games):,}; matchups: {len(matchups):,}')
print('Season range:', matchups.season.min(), 'to', matchups.season.max())
print('Stage 2 files loaded successfully.')

Mounted at /content/drive
Team-game rows: 36,600; matchups: 18,300
Season range: 2004 to 2025
Stage 2 files loaded successfully.


## 1. Download historical team subdivision membership



In [ ]:
%pip install -q pyarrow requests

import requests

MEMBERSHIP_URL = (
    'https://github.com/sportsdataverse/sportsdataverse-data/'
    'releases/download/cfb_groups/cfb_team_group_seasons.parquet'
)
MEMBERSHIP_FILE = REFERENCE_DIR / 'cfb_team_group_seasons.parquet'

if not MEMBERSHIP_FILE.is_file():
    print('Downloading historical team membership reference...')
    response = requests.get(MEMBERSHIP_URL, timeout=120)
    response.raise_for_status()
    # Catch an HTML error page instead of silently caching it as Parquet.
    if response.content[:100].lstrip().lower().startswith(b'<!doctype html'):
        raise ValueError('Got an HTML page rather than Parquet data.')
    MEMBERSHIP_FILE.write_bytes(response.content)

membership_raw = pd.read_parquet(MEMBERSHIP_FILE)
required = {'season', 'team_name', 'subdivision_id'}
assert required.issubset(membership_raw.columns), membership_raw.columns.tolist()
membership_raw = membership_raw.loc[
    membership_raw['season'].between(2004, 2025)
].copy()
membership_raw['subdivision'] = (
    membership_raw['subdivision_id'].astype('string')
    .str.removeprefix('cfb:').str.lower()
)

print('Historical membership rows:', len(membership_raw))
print('Reference years:', membership_raw.season.min(), 'to', membership_raw.season.max())
print('Available subdivision labels:')
display(membership_raw['subdivision'].value_counts(dropna=False).to_frame('team_seasons'))
display(membership_raw[['season','team_name','subdivision']].head(8))

Historical membership rows: 10718
Reference years: 2004 to 2025
Available subdivision labels:


,team_seasons
subdivision,
d3,2977
fbs,2773
fcs,2752
d2,2216


,season,team_name,subdivision
22665,2004,Boston College,fbs
22666,2004,Holy Cross,fcs
22667,2004,Harvard,fcs
22668,2004,Northeastern,fcs
22669,2004,Massachusetts,fcs
22670,2004,Towson,fcs
22671,2004,Arizona,fbs
22672,2004,Maryland,fbs


## 2. Match names safely and audit uncertain identities


In [ ]:
def normalize_name(value):
    if pd.isna(value):
        return ''
    s = unicodedata.normalize('NFKD', str(value))
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = s.casefold().replace('&', ' and ')
    s = re.sub(r'[^a-z0-9]+', ' ', s)
    return ' '.join(s.split())

# Only manually verified changes belong here.
# Example format: (2025, 'Kaggle team name'): 'Exact reference team name'
# Known team-name differences to check
ALIAS_CANDIDATES = {
    "Miami (FL)": ["Miami"],
    "Hawaii": ["Hawai'i"],
    "Appalachian State": ["App State"],
    "USF": ["South Florida"],
    "UL-Lafayette": ["Louisiana", "Louisiana-Lafayette"],
    "FIU": ["Florida International", "Florida Intl."]
}

# Additional team-name aliases to verify
ALIAS_CANDIDATES.update({
    "Connecticut": ["UConn"],
    "Sam Houston State": ["Sam Houston"],
    "Tennessee-Martin": ["UT Martin", "Tennessee Martin"],
    "Citadel": ["The Citadel"],
    "LIU": ["Long Island", "Long Island University"],
    "Southeastern Louisiana": ["SE Louisiana"],
    "Nicholls State": ["Nicholls"],
    "Virginia Military": ["VMI"]
})

# Additional FCS naming differences
ALIAS_CANDIDATES.update({
    "Albany": ["UAlbany"],
    "Cal Poly SLO": ["Cal Poly"],
    "Central Conn. State": ["Central Connecticut"],
    "Grambling State": ["Grambling"],
    "SE Missouri State": ["Southeast Missouri State"],
    "McNeese State": ["McNeese"],
})


# Reference names available in each season
reference_names = (
    membership_raw.groupby("season")["team_name"]
    .agg(set)
    .to_dict()
)

MANUAL_ALIASES = {}

for season, team in team_games[
    ["season", "team"]
].drop_duplicates().itertuples(index=False, name=None):

    candidates = ALIAS_CANDIDATES.get(team, [])
    valid = [
        name for name in candidates
        if name in reference_names.get(season, set())
    ]

    if len(valid) == 1:
        MANUAL_ALIASES[(int(season), team)] = valid[0]

    elif len(valid) > 1:
        print(f"Ambiguous alias: {season}, {team}: {valid}")

print("Aliases verified:", len(MANUAL_ALIASES))

# List of unique team/season pairs present in the Kaggle dataset.
kaggle_team_seasons = (
    team_games[['season', 'team']].drop_duplicates().copy()
)
kaggle_team_seasons['lookup_name'] = kaggle_team_seasons.apply(
    lambda row: MANUAL_ALIASES.get((int(row['season']), row['team']), row['team']), axis=1
)
kaggle_team_seasons['match_key'] = kaggle_team_seasons['lookup_name'].map(normalize_name)

reference = membership_raw[['season', 'team_name', 'subdivision', 'subdivision_id']].copy()
reference['match_key'] = reference['team_name'].map(normalize_name)

# A normalized reference name must identify exactly one team per season.
colliding_keys = reference.duplicated(['season','match_key'], keep=False)
if colliding_keys.any():
    print('Reference-name collisions (excluded from automatic matching):')
    display(reference.loc[colliding_keys].sort_values(['season','match_key']).head(30))
reference_unique = reference.loc[~colliding_keys].copy()

team_membership = kaggle_team_seasons.merge(
    reference_unique[['season','match_key','team_name','subdivision','subdivision_id']],
    on=['season','match_key'], how='left', validate='one_to_one'
)
team_membership['subdivision'] = team_membership['subdivision'].fillna('unknown')
team_membership['matched'] = team_membership['subdivision'].ne('unknown')
team_membership['used_manual_alias'] = team_membership.apply(
    lambda r: (int(r['season']), r['team']) in MANUAL_ALIASES, axis=1
)

# Make name suggestions for review, but never apply them automatically.
ref_names_by_year = {int(y): ref.team_name.dropna().tolist()
                     for y, ref in reference.groupby('season')}
def suggest_reference(row):
    possibilities = ref_names_by_year.get(int(row['season']), [])
    close = get_close_matches(row['lookup_name'], possibilities, n=3, cutoff=0.45)
    return ' | '.join(close)

unmatched = team_membership.loc[~team_membership['matched']].copy()
# Create a consistent review-table schema even if every name matches.
unmatched['suggested_reference_names'] = pd.Series(index=unmatched.index, dtype='string')
appearances = (team_games.groupby(['season','team']).size()
               .rename('recorded_appearances').reset_index())
unmatched = unmatched.merge(appearances, on=['season','team'], how='left')
if len(unmatched):
    unmatched['suggested_reference_names'] = unmatched.apply(suggest_reference, axis=1)
    unmatched = unmatched.sort_values('recorded_appearances', ascending=False)

print('Distinct Kaggle team-seasons:', len(team_membership))
print('Unmatched team-seasons:', len(unmatched))
print('Match rate:', f"{team_membership['matched'].mean():.1%}")
print('Top unmatched team-season pairs (review these before modeling):')
display(unmatched[['season','team','recorded_appearances',
                   'suggested_reference_names']].head(35))

Aliases verified: 374
Distinct Kaggle team-seasons: 4444
Unmatched team-seasons: 23
Match rate: 99.5%
Top unmatched team-season pairs (review these before modeling):


,season,team,recorded_appearances,suggested_reference_names
22,2025,St. Francis (PA),2,Saint Francis | St. Thomas (MN) | St. Lawrence
21,2024,St. Francis (PA),2,Saint Francis | St. Thomas (MN) | St. Lawrence
0,2007,Elon University,1,Long Island University | Liberty
3,2011,Elon University,1,Long Island University | Liberty
4,2012,Elon University,1,Long Island University | Liberty
1,2009,Elon University,1,Long Island University | Liberty
2,2010,Elon University,1,Long Island University | Liberty
6,2014,Elon University,1,Tiffin University | Long Island University | F...
5,2013,Elon University,1,Long Island University | Liberty
7,2015,Elon University,1,Tiffin University | Long Island University | F...


## 3. Label games; inspect FCS and unknown coverage


In [ ]:
lookup = team_membership[['season','team','subdivision']].copy()
side_home = lookup.rename(columns={'team':'home', 'subdivision':'home_subdivision'})
side_away = lookup.rename(columns={'team':'away', 'subdivision':'away_subdivision'})

eligibility = matchups[['game_id','season','date','home','away','neutral']].copy()
eligibility = eligibility.merge(side_home, on=['season','home'], how='left', validate='many_to_one')
eligibility = eligibility.merge(side_away, on=['season','away'], how='left', validate='many_to_one')
assert eligibility['game_id'].is_unique and len(eligibility) == len(matchups)

eligibility['home_subdivision'] = eligibility['home_subdivision'].fillna('unknown')
eligibility['away_subdivision'] = eligibility['away_subdivision'].fillna('unknown')
fbs_home = eligibility['home_subdivision'].eq('fbs')
fbs_away = eligibility['away_subdivision'].eq('fbs')
fcs_home = eligibility['home_subdivision'].eq('fcs')
fcs_away = eligibility['away_subdivision'].eq('fcs')
unknown = eligibility[['home_subdivision','away_subdivision']].eq('unknown').any(axis=1)

eligibility['game_category'] = np.select(
    [unknown,
     fbs_home & fbs_away,
     (fbs_home & fcs_away) | (fbs_away & fcs_home),
     fcs_home & fcs_away],
    ['unknown_membership', 'FBS_vs_FBS','FBS_vs_FCS','FCS_vs_FCS'],
    default='other_divisions'
)
eligibility['primary_fbs_fbs'] = eligibility['game_category'].eq('FBS_vs_FBS')

print('Matchups by category:')
display(eligibility['game_category'].value_counts(dropna=False).to_frame('games'))
print('Fraction of games eligible for primary FBS model:',
      f"{eligibility['primary_fbs_fbs'].mean():.1%}")
print('Examples from seasons when schools may have changed subdivisions:')
example_schools = ['Appalachian State', 'James Madison', 'Liberty',
                   'Jacksonville State', 'Sam Houston', 'Kennesaw State']
display(team_membership.loc[
    team_membership['team'].isin(example_schools),
    ['season','team','subdivision']
].sort_values(['team','season']).tail(35))
print('Sample games with unresolved classification:')
display(eligibility.loc[eligibility.game_category.eq('unknown_membership'),
     ['season','away','home','away_subdivision','home_subdivision']].head(20))


Matchups by category:


,games
game_category,
FBS_vs_FBS,16169
FBS_vs_FCS,2105
unknown_membership,25
other_divisions,1


Fraction of games eligible for primary FBS model: 88.4%
Examples from seasons when schools may have changed subdivisions:


,season,team,subdivision
1581,2017,James Madison,fcs
1582,2018,James Madison,fcs
1583,2019,James Madison,fcs
1584,2022,James Madison,fbs
1585,2023,James Madison,fbs
1586,2024,James Madison,fbs
1587,2025,James Madison,fbs
1632,2018,Kennesaw State,fcs
1633,2019,Kennesaw State,fcs
1634,2021,Kennesaw State,fcs


Sample games with unresolved classification:


,season,away,home,away_subdivision,home_subdivision
2277,2007,Elon University,USF,unknown,fbs
3990,2009,Elon University,Wake Forest,unknown,fbs
4675,2010,Elon University,Duke,unknown,fbs
5494,2011,Elon University,Vanderbilt,unknown,fbs
6273,2012,Elon University,North Carolina,unknown,fbs
6474,2012,Northwestern Oklahoma State,UTSA,unknown,fbs
7110,2013,Elon University,Georgia Tech,unknown,fbs
7984,2014,Elon University,Duke,unknown,fbs
8805,2015,Elon University,Wake Forest,unknown,fbs
9802,2016,Elon University,Charlotte,unknown,fbs


## 4. Declare chronological validation folds and test period



In [ ]:
VALIDATION_YEARS = list(range(2019, 2024))
TEST_YEARS = [2024, 2025]
FOLDS = [
    {'fold': f'val_{year}', 'train_seasons': [2004, year-1],
     'validation_season': year}
    for year in VALIDATION_YEARS
]

eligibility['partition'] = np.select(
    [eligibility['season'].le(2018),
     eligibility['season'].between(2019, 2023),
     eligibility['season'].between(2024, 2025)],
    ['initial_training','validation_era','final_test'], default='out_of_scope'
)
eligibility['validation_fold'] = np.where(
    eligibility['season'].isin(VALIDATION_YEARS),
    'val_' + eligibility['season'].astype(str),
    ''
)

assert (eligibility['partition'] != 'out_of_scope').all()
assert eligibility.loc[eligibility.partition.eq('final_test'),'season'].isin(TEST_YEARS).all()
assert eligibility['game_id'].is_unique
assert len(FOLDS) == 5

print('All games by partition / category:')
display(pd.crosstab(eligibility.partition, eligibility.game_category))
print('Primary FBS–FBS games by season / partition:')
display(eligibility.loc[eligibility.primary_fbs_fbs]
        .groupby(['season','partition']).size().rename('games').reset_index())
print('Expanding-window folds:')
display(pd.DataFrame(FOLDS))

All games by partition / category:


game_category,FBS_vs_FBS,FBS_vs_FCS,other_divisions,unknown_membership
partition,,,,
final_test,1606,241,0,6
initial_training,10917,1368,1,12
validation_era,3646,496,0,7


Primary FBS–FBS games by season / partition:


,season,partition,games
0,2004,initial_training,661
1,2005,initial_training,664
2,2006,initial_training,714
3,2007,initial_training,712
4,2008,initial_training,717
5,2009,initial_training,714
6,2010,initial_training,718
7,2011,initial_training,715
8,2012,initial_training,732
9,2013,initial_training,737


Expanding-window folds:


,fold,train_seasons,validation_season
0,val_2019,"[2004, 2018]",2019
1,val_2020,"[2004, 2019]",2020
2,val_2021,"[2004, 2020]",2021
3,val_2022,"[2004, 2021]",2022
4,val_2023,"[2004, 2022]",2023


## 5. Save eligibility manifest and audit to Drive



In [ ]:
OUTPUT_ELIGIBILITY = PROCESSED_DIR / 'game_eligibility_and_splits.csv'
OUTPUT_MEMBERSHIP = PROCESSED_DIR / 'team_season_subdivisions.csv'
OUTPUT_UNMATCHED = REFERENCE_DIR / 'unmatched_team_names_review.csv'
OUTPUT_MANIFEST = PROCESSED_DIR / 'split_manifest.json'

eligibility.to_csv(OUTPUT_ELIGIBILITY, index=False)
team_membership.to_csv(OUTPUT_MEMBERSHIP, index=False)
unmatched.to_csv(OUTPUT_UNMATCHED, index=False)
manifest = {
    'training_start_season': 2004,
    'validation_folds': FOLDS,
    'test_seasons': TEST_YEARS,
    'primary_population': 'both teams verified FBS in game season',
    'unknown_membership_policy': 'not eligible until resolved',
    'reference_url': MEMBERSHIP_URL,
    'reference_file': str(MEMBERSHIP_FILE),
    'membership_source': 'SportsDataverse season-specific team group memberships',
    'manual_name_aliases': [
        {'season':int(y),'kaggle_name':k,'reference_name':v}
        for (y,k),v in MANUAL_ALIASES.items()
    ]
}
OUTPUT_MANIFEST.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved:')
for p in [OUTPUT_ELIGIBILITY, OUTPUT_MEMBERSHIP, OUTPUT_UNMATCHED, OUTPUT_MANIFEST]:
    print(' -', p)
print('Unknown-classification games:', int(eligibility.game_category.eq('unknown_membership').sum()))
print('Review unmatched names before relying on the primary population.')

Saved:
 - /content/drive/MyDrive/CFB_Upset_Project/data/processed/game_eligibility_and_splits.csv
 - /content/drive/MyDrive/CFB_Upset_Project/data/processed/team_season_subdivisions.csv
 - /content/drive/MyDrive/CFB_Upset_Project/data/reference/unmatched_team_names_review.csv
 - /content/drive/MyDrive/CFB_Upset_Project/data/processed/split_manifest.json
Unknown-classification games: 25
Review unmatched names before relying on the primary population.
